In [1]:
# Cài Unsloth cho Kaggle (tag "kaggle-new" đúng cho môi trường Kaggle)
!pip install "unsloth[kaggle-new]" --upgrade --quiet

print("✅ Cài đặt hoàn tất! Vào Menu > Run > Restart Session, rồi chạy từ Cell 2.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 108.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 98.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 102.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# import IPython
# IPython.Application.instance().kernel.do_shutdown(True)


In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch
from huggingface_hub import login
from datasets import load_dataset
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from trl import SFTTrainer, SFTConfig

from kaggle_secrets import UserSecretsClient
try:
    user_secrets = UserSecretsClient()
    VaccineNLP_TOKEN = user_secrets.get_secret("VaccineNLP")
    login(token=VaccineNLP_TOKEN)
    print("✅ Đăng nhập HuggingFace thành công!")
except Exception as e:
    print(f"⚠️ Không tìm thấy VaccineNLP: {e}")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
✅ Đăng nhập HuggingFace thành công!


In [4]:
# FastModel (không phải FastLanguageModel) — đúng cho Gemma 4 multimodal
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E4B-it",  # Tên đúng trên HuggingFace
    max_seq_length = 1024,
    dtype = None,           # Unsloth tự detect BF16/FP16
    load_in_4bit = True,
)

# LoRA — chỉ tune language layers, bỏ qua vision/audio tower
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers   = False,  # Task text-only, không cần vision
    finetune_language_layers = True,
    finetune_attention_modules = True,
    finetune_mlp_modules     = True,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # Unsloth chống OOM tốt hơn HF
    random_state = 3407,
)
model.print_trainable_parameters()
print("✅ Load model hoàn tất!")


==((====))==  Unsloth 2026.4.8: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

trainable params: 36,700,160 || all params: 8,032,856,608 || trainable%: 0.4569
✅ Load model hoàn tất!


In [5]:
TRAIN_PATH      = "/kaggle/input/datasets/inhlqunhphng/vaccinenlp-clean-data/05_model_ready/train_v2_seg.jsonl"
TEST_PATH       = "/kaggle/input/datasets/inhlqunhphng/vaccinenlp-clean-data/03_processed/benchmark_test_set.jsonl"
MODELS_SAVE_DIR = "/kaggle/working/gemma_qlora_xai"
os.makedirs(MODELS_SAVE_DIR, exist_ok=True)

for path in [TRAIN_PATH, TEST_PATH]:
    print(("✅" if os.path.exists(path) else "❌ KHÔNG TÌM THẤY") + f": {path}")

# Dùng "gemma-4" (không phải "gemma-4-thinking") vì:
# Task classification không cần thinking tokens → tiết kiệm VRAM
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

misinfo_map   = {0: "Không liên quan", 1: "Tin giả", 2: "Chính xác"}
stance_map    = {0: "Ủng hộ", 1: "Phản đối", 2: "Trung lập"}
sentiment_map = {0: "Tiêu cực", 1: "Trung tính", 2: "Tích cực"}

def format_prompt(row):
    text = row.get('text_cleaned') or row.get('text') or str(list(row.values())[0])
    reasoning = row.get('llm_reasoning', 'Phân tích dựa trên ngữ cảnh.')
    ids = row.get('standardized_ids') or [0, 3, 1]

    convo = [
        {"role": "user", "content": (
            f"You are an Explainable AI in Public Health. Analyze the text, "
            f"provide your reasoning first, then the structured labels.\n\nVăn bản: {text}"
        )},
        {"role": "model", "content": (
            f"Lý luận: {reasoning}\n"
            f"Kết quả: {misinfo_map.get(ids[0], 'Không liên quan')} | "
            f"{stance_map.get(ids[1], 'Không rõ')} | "
            f"{sentiment_map.get(ids[2], 'Trung tính')}"
        )}
    ]
    # removeprefix('<bos>') theo hướng dẫn Unsloth — Processor sẽ tự thêm lại khi train
    return {"text": tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False).removeprefix('<bos>')}

print("Đang tải và chia tập dữ liệu (90/10)...")
# Load raw dataset first to split
raw_train_ds = load_dataset("json", data_files=TRAIN_PATH, split="train")
split_ds = raw_train_ds.train_test_split(test_size=0.1, seed=42)

train_ds = split_ds["train"].map(format_prompt)
eval_ds  = split_ds["test"].map(format_prompt)
test_ds  = load_dataset("json", data_files=TEST_PATH, split="train").map(format_prompt)

print(f"✅ Train: {len(train_ds)} mẫu | Val: {len(eval_ds)} mẫu | Test: {len(test_ds)} mẫu")
print(train_ds[0]['text'][:300] + "...")


✅: /kaggle/input/datasets/inhlqunhphng/vaccinenlp-clean-data/05_model_ready/train_v2_seg.jsonl
✅: /kaggle/input/datasets/inhlqunhphng/vaccinenlp-clean-data/03_processed/benchmark_test_set.jsonl
Đang tải và chia tập dữ liệu (90/10)...


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1503 [00:00<?, ? examples/s]

Map:   0%|          | 0/167 [00:00<?, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/186 [00:00<?, ? examples/s]

✅ Train: 1503 mẫu | Val: 167 mẫu | Test: 186 mẫu
<|turn>user
You are an Explainable AI in Public Health. Analyze the text, provide your reasoning first, then the structured labels.

Văn bản: Bác sĩ ơi con bị cho liếm lỗ tai có bị gì không ạ<turn|>
<|turn>model
Lý luận: Phân tích dựa trên ngữ cảnh.
Kết quả: Chính xác | Trung lập | Trung tính<turn|>...


In [6]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = eval_ds,     # Tắt eval để tránh OOM trên T4
    args = SFTConfig(
        output_dir = MODELS_SAVE_DIR,
        dataset_text_field = "text",
        max_length = 1024,
        per_device_train_batch_size = 2,   # Unsloth cho phép batch=2 thay vì 1
        per_device_eval_batch_size = 1, # THÊM DÒNG NÀY
        eval_accumulation_steps = 1,    # THÊM DÒNG NÀY
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        logging_steps = 10,
        save_strategy = "epoch",
        eval_strategy = "steps",
        eval_steps = 50,
        optim = "adamw_8bit",
        bf16 = torch.cuda.is_bf16_supported(),
        fp16 = not torch.cuda.is_bf16_supported(),
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
    ),
)

# Chỉ tính loss trên phần model trả lời, không tính loss trên câu hỏi
# → Tăng chất lượng fine-tuning đáng kể
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part    = "<|turn>model\n",
)

print("🚀 Bắt đầu huấn luyện...")
trainer.train()

model.save_pretrained(f"{MODELS_SAVE_DIR}/final_model")
tokenizer.save_pretrained(f"{MODELS_SAVE_DIR}/final_model")
print(f"💾 Đã lưu model tại: {MODELS_SAVE_DIR}/final_model")


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/1503 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/167 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/1503 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/1503 [00:00<?, ? examples/s]

Unsloth: Removed 52 out of 1503 samples from train_dataset where all labels were -100 (no response found after truncation). This prevents NaN loss during training.


...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,451 | Num Epochs = 2 | Total steps = 364
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 36,700,160 of 8,032,856,608 (0.46% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
50,0.086273,3.324119
100,0.078717,3.404793
150,0.065771,3.562285
200,0.055634,3.543759
250,0.046468,3.645554
300,0.052324,3.562731
350,0.047747,3.616433
364,0.049231,3.628867


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/gemma_qlora_xai/checkpoint-182/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/gemma_qlora_xai/checkpoint-364/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/gemma_qlora_xai/final_model/tokenizer_config.json.


💾 Đã lưu model tại: /kaggle/working/gemma_qlora_xai/final_model


In [7]:
import shutil
from IPython.display import FileLink

print("Đang nén mô hình...")
# Nén thư mục final_model thành file gemma_unsloth_model.zip
shutil.make_archive('gemma_unsloth_model', 'zip', '/kaggle/working/gemma_qlora_xai/final_model')
print("✅ Hoàn tất! Bấm vào link bên dưới để tải về:")

# Hiển thị link tải
display(FileLink('gemma_unsloth_model.zip'))

Đang nén mô hình...
✅ Hoàn tất! Bấm vào link bên dưới để tải về:


/kaggle/working/gemma_unsloth_model.zip

In [8]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
VaccineNLP_TOKEN = user_secrets.get_secret("VaccineNLP")

repo_id = "quynhphuong1209/gemma-4-E4B-unsloth-vaccine-xai"

print(f"📤 Đang đẩy lên {repo_id}...")
model.push_to_hub(repo_id, token=VaccineNLP_TOKEN)
tokenizer.push_to_hub(repo_id, token=VaccineNLP_TOKEN)
print(f"🎉 Hoàn tất! https://huggingface.co/{repo_id}")


📤 Đang đẩy lên quynhphuong1209/gemma-4-E4B-unsloth-vaccine-xai...


README.md:   0%|          | 0.00/577 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/quynhphuong1209/gemma-4-E4B-unsloth-vaccine-xai


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpqyoqsike/tokenizer_config.json.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


🎉 Hoàn tất! https://huggingface.co/quynhphuong1209/gemma-4-E4B-unsloth-vaccine-xai


In [9]:
# ==========================================
# [CELL TEST NÓNG] - V2: Kỷ Luật Thép (Taxonomy Enforced)
# ==========================================
from transformers import TextStreamer
import torch

stress_tests = [
    "tiêm vs xog con t sốt cmn li bì, bs bảo bìh thg nhg trên mạng đầy vụ tự kỷ ròi, đm bọnn chóp bu lùa gà vkl =)))",
    "T.j.ê.m v.xln là thuyết â/m m/ư/u của Bjn Gết để cấy chj.p 5G dkyển con ng nhé mấy con giời. Tỉnh mộng đi đừg nạp mạng cho Big Fama nưã 🛑",
    "mới bú con as cha zenca bầm dập văng cmn nết 🤡 nma thui ráng để lấy thẻ xanh đi đu idol Hàn Xẻng. done mũi 2 nhóe mn!",
    "Bácsĩơi,bénhàem tiêmmũi5trong1 bịsưng cụcđỏ hơn1tuần rồicó saokg ạ??? Emsợquá bôidầugió mà kgđỡ :(",
    "Ai ns chích ngừa êm ru hông sốt thì bốc phét nhé. T vật vã 2 ngày tưởng đi chầu ông bà vải cmnr. Nma ko chích thì h chăc đag bú bình oxy cmnr. 10 điểm ko có nhưng 👍"
]

model.config.use_cache = True 

print("🚀 ĐANG KHỞI CHẠY HỆ THỐNG XAI (CHẾ ĐỘ KỶ LUẬT THÉP)...\n")

for i, text in enumerate(stress_tests):
    print(f"--- TEST CASE {i+1} ---")
    print(f"📥 Input: {text}")
    
    # 1. IN-CONTEXT TAXONOMY PROMPT: Nhắc lại barem chấm điểm
    user_prompt = (
        f"Phân tích y tế: {text}\n\n"
        "Nhiệm vụ: Trình bày chuỗi lý luận (Chain-of-Thought) ngắn gọn và gán nhãn theo đúng 3 trục sau:\n"
        "1. Misinformation: Tin giả | Chính xác | Không liên quan\n"
        "2. Stance: Ủng hộ | Phản đối | Trung lập | Không rõ\n"
        "3. Sentiment: Tiêu cực | Trung tính | Tích cực\n\n"
        "Bắt buộc trả về đúng định dạng:\nLý luận: [Nội dung]\nKết quả: [Misinfo] | [Stance] | [Sentiment]"
    )
    
    messages = [
        {"role": "user", "content": [{"type": "text", "text": user_prompt}]}
    ]
    
    # Render prompt
    prompt_str = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    # 2. FORCED PREFILLING
    forced_prompt = prompt_str + "Lý luận: "
    inputs = tokenizer(text=forced_prompt, return_tensors="pt").to("cuda")
    
    print("🧠 Model Reasoning:")
    print("Lý luận: ", end="") 
    
    from transformers import TextStreamer
    text_streamer = TextStreamer(tokenizer, skip_prompt=True) 
    
    _ = model.generate(
        **inputs, 
        streamer = text_streamer,
        # NHỮNG THAY ĐỔI CỐT LÕI NẰM Ở ĐÂY:
        max_new_tokens = 512,        # 1. THẢ XÍCH: Tăng từ 200 lên 512 tokens để nó có đủ hơi nói hết câu.
        temperature = 0.2,           # 2. Nhiệt độ: 0.2 đủ thấp để giữ logic, đủ cao để không bị kẹt lặp từ.
        repetition_penalty = 1.15,   # 3. Phạt lặp từ: Tăng lên một chút để ép nó nhanh chóng chốt "Kết quả" thay vì kể lể mãi.
        pad_token_id = tokenizer.eos_token_id, # Đảm bảo mô hình biết lúc nào nên tự động dừng (Stop generation).
        use_cache = True
    )
    print("\n" + "="*70 + "\n")

🚀 ĐANG KHỞI CHẠY HỆ THỐNG XAI (CHẾ ĐỘ KỶ LUẬT THÉP)...

--- TEST CASE 1 ---
📥 Input: tiêm vs xog con t sốt cmn li bì, bs bảo bìh thg nhg trên mạng đầy vụ tự kỷ ròi, đm bọnn chóp bu lùa gà vkl =)))
🧠 Model Reasoning:
Lý luận: 
1. **Phân tích nội dung:** Người dùng đang bày tỏ sự lo lắng và tức giận về việc con bị sốt, li bì sau khi tiêm chủng. Họ trích dẫn thông tin tiêu cực từ mạng xã hội về nguy cơ tự kỷ liên quan đến tiêm chủng, và kết luận rằng bác sĩ đang "lùa gà".
2. **Đánh giá tính chính xác (Misinformation):** Việc quy kết trực tiếp giữa vắc-xin và tự kỷ là một thuyết âm mưu đã bị khoa học bác bỏ rộng rãi. Các cơ quan y tế lớn trên thế giới khẳng định không có mối liên hệ nhân quả giữa vắc-xin và tự kỷ. Do đó, nhận định này thuộc loại tin giả (Misinformation).
3. **Đánh giá thái độ (Stance):** Người dùng đang hoàn toàn phản đối hành động tiêm chủng và bác sĩ đưa ra lời khuyên.
4. **Đánh giá cảm xúc (Sentiment):** Ngôn ngữ sử dụng ("đm", "bọnn chóp bu lùa gà") thể hiện sự giận dữ